# Task 7: Low-Rank Adaptation (LoRA) Matrix Projection Fine-Tuning of a 3B Model

## Objective

To implement parameter-efficient fine-tuning using Low-Rank Adaptation (LoRA), where only low-rank adapter matrices are trained while the original model parameters remain frozen.

## Technologies / Tools Used

- Python
- PyTorch
- Hugging Face PEFT
- Transformers
- CUDA Toolkit
- Google Colab

## Formula

### LoRA Weight Update

\[
W' = W + \frac{\alpha}{r}BA
\]

where \(A\) and \(B\) are low-rank matrices and \(r \leq 8\).

### Scaling Factor

\[
s = \frac{\alpha}{r}
\]

Only the LoRA matrices are updated during fine-tuning.

In [6]:
# Install compatible versions

!pip -q install -U "torchao>=0.16.0" peft transformers accelerate

import torch
import torch.nn as nn

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 29.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 44.9 MB/s eta 0:00:00
PyTorch: 2.11.0+cpu
CUDA available: False


## Step 1: Create a Custom LoRA Projection

Create low-rank matrices A and B parallel to the original projection weight.

In [2]:
class LoRALinear(nn.Module):

    def __init__(self, in_features, out_features, r=4, alpha=8):
        super().__init__()

        self.base = nn.Linear(
            in_features,
            out_features
        )

        self.A = nn.Parameter(
            torch.randn(r, in_features) * 0.01
        )

        self.B = nn.Parameter(
            torch.zeros(out_features, r)
        )

        self.scaling = alpha / r

        # Freeze original weights

        self.base.weight.requires_grad = False
        self.base.bias.requires_grad = False

    def forward(self, x):

        base_output = self.base(x)

        lora_output = (
            x @ self.A.T @ self.B.T
        ) * self.scaling

        return base_output + lora_output


model = LoRALinear(
    16,
    16,
    r=4,
    alpha=8
)

print("LoRA layer created.")
print("Scaling:", model.scaling)
print("Rank:", model.A.shape[0])

LoRA layer created.
Scaling: 2.0
Rank: 4


## Step 2: Verify Trainable Parameters

Freeze the base model parameters and verify that only the low-rank matrices A and B are trainable.

In [3]:
for name, parameter in model.named_parameters():

    print(
        name,
        "Trainable:",
        parameter.requires_grad,
        "Shape:",
        tuple(parameter.shape)
    )

trainable = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

total = sum(
    p.numel()
    for p in model.parameters()
)

print("\nTrainable parameters:", trainable)
print("Total parameters:", total)
print(
    "Trainable percentage:",
    round(100 * trainable / total, 2),
    "%"
)

A Trainable: True Shape: (4, 16)
B Trainable: True Shape: (16, 4)
base.weight Trainable: False Shape: (16, 16)
base.bias Trainable: False Shape: (16,)

Trainable parameters: 128
Total parameters: 400
Trainable percentage: 32.0 %


## Step 3: Train Only the LoRA Parameters

Use a small synthetic domain-specific dataset and optimize only the LoRA adapter matrices.

In [4]:
# Small synthetic dataset

X = torch.randn(32, 16)

# Target output

Y = torch.randn(32, 16)

optimizer = torch.optim.Adam(
    [model.A, model.B],
    lr=0.01
)

loss_function = nn.MSELoss()

for epoch in range(20):

    optimizer.zero_grad()

    output = model(X)

    loss = loss_function(
        output,
        Y
    )

    loss.backward()

    optimizer.step()

    if epoch % 5 == 0:
        print(
            f"Epoch {epoch}: Loss = {loss.item():.4f}"
        )

print("LoRA training completed.")

Epoch 0: Loss = 1.3537
Epoch 5: Loss = 1.3187
Epoch 10: Loss = 1.2177
Epoch 15: Loss = 1.1112
LoRA training completed.


## Step 4: LoRA with Hugging Face PEFT

PEFT can automatically insert LoRA adapters into Transformer attention projection layers while keeping the original model frozen.

In [7]:
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

# Small model for fast Colab execution

model_name = "sshleifer/tiny-gpt2"

base_model = AutoModelForCausalLM.from_pretrained(
    model_name
)

lora_config = LoraConfig(
    r=4,
    lora_alpha=8,
    lora_dropout=0.05,
    target_modules=["c_attn"],
    bias="none",
    task_type="CAUSAL_LM"
)

peft_model = get_peft_model(
    base_model,
    lora_config
)

peft_model.print_trainable_parameters()

Loading weights:   0%|          | 0/29 [00:00<?, ?it/s]

[transformers] GPT2LMHeadModel LOAD REPORT from: sshleifer/tiny-gpt2
Key                                   | Status     |  | 
--------------------------------------+------------+--+-
transformer.h.{0, 1}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


trainable params: 64 || all params: 102,778 || trainable%: 0.0623


/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/layer.py:2504: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  )


## Conclusion

LoRA fine-tuning was successfully implemented using custom low-rank projection matrices and Hugging Face PEFT. The base model parameters were frozen, while only the low-rank adapter parameters were optimized. The scaling factor \(\alpha/r\) was applied to control the LoRA update magnitude.